# Data Acquisition for FinTech-AI-MIT Paper (cleaned)
## Downloads everything Colab can fetch automatically, lists the rest

**How to run:** Runtime → Run all. Outputs land in `/content/data/raw/` and a
merged panel in `/content/data/processed/panel.parquet`.

**What changed in this cleaned version:**
- Every source is trimmed to **only the columns the panel needs** before saving —
  no more hundred-column raw dumps polluting the merge.
- **Findex** is filtered to country-level totals (`group == "all"`) and the four
  F-index components (+ account ownership), for **all waves**, and is now merged
  into the panel.
- **GARI 2024** and **IMF AIPI** are single-year snapshots; they now merge on
  `iso3` only (time-invariant columns `gari_total_2024`, `aipi_*_2023`), instead
  of attaching to a single year-row and looking missing everywhere else.
- **Income classification** merges keep only the codes/dummies, not names/labels.
- WGI and CLASS.xlsx **auto-download cells now run before parsing** (manual
  upload is only a fallback).
- All debug cells (`!find`, `%whos`, repeated `drive.mount`, raw-sheet previews,
  broken `df.to_csv`) removed.
- The final panel is clamped to `YEAR_MIN–YEAR_MAX`, real ISO3 countries only,
  deduplicated on `(iso3, year)`.


## 1. Setup

In [1]:
!pip install -q wbdata country_converter pycountry pyarrow openpyxl

import gzip, json, logging, os, time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Working directories (subfolders for manual uploads are created up-front)
RAW = Path("/content/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = Path("/content/data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)
for sub in ["oxford_gari", "wgi", "imf_aipi", "wb_income", "ilostat", "findex"]:
    (RAW/sub).mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("data")

YEAR_MIN, YEAR_MAX = 2010, 2024
print("Setup complete. Outputs will land in /content/data/")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 32.0 MB/s eta 0:00:00
Setup complete. Outputs will land in /content/data/


**Optional — persist to Google Drive.** Uncomment and run the cell below if you
want outputs to survive the Colab session. Everything downstream just uses
`RAW`/`PROCESSED`, so nothing else changes.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')
# RAW = Path("/content/drive/MyDrive/fintech_ai_mit/data/raw")
# PROCESSED = Path("/content/drive/MyDrive/fintech_ai_mit/data/processed")
# RAW.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
# for sub in ["oxford_gari", "wgi", "imf_aipi", "wb_income", "ilostat", "findex"]:
#     (RAW/sub).mkdir(parents=True, exist_ok=True)


## 2. Country code harmonization

We convert all sources to ISO 3166-1 alpha-3.

In [3]:
import country_converter as coco
import logging
logging.getLogger("country_converter").setLevel(logging.ERROR)

_CC = coco.CountryConverter()

MANUAL_ISO3 = {
    "kosovo": "XKX", "west bank and gaza": "PSE", "taiwan, china": "TWN",
    "korea, rep.": "KOR", "korea, dem. people's rep.": "PRK",
    "lao pdr": "LAO", "egypt, arab rep.": "EGY", "iran, islamic rep.": "IRN",
    "venezuela, rb": "VEN", "yemen, rep.": "YEM", "turkiye": "TUR",
    "cote d'ivoire": "CIV", "eswatini": "SWZ", "czechia": "CZE",
    "north macedonia": "MKD", "cabo verde": "CPV", "myanmar": "MMR",
    "congo, dem. rep.": "COD", "congo, rep.": "COG",
}

AGGREGATES = {
    "world", "high income", "low income", "middle income",
    "lower middle income", "upper middle income",
    "low & middle income", "least developed countries: un classification",
    "heavily indebted poor countries (hipc)",
    "ida total", "ida blend", "ida only", "ida & ibrd total",
    "ibrd only", "fragile and conflict affected situations",
    "east asia & pacific", "east asia & pacific (excluding high income)",
    "east asia & pacific (ida & ibrd countries)",
    "europe & central asia", "europe & central asia (excluding high income)",
    "europe & central asia (ida & ibrd countries)",
    "european union", "euro area",
    "latin america & caribbean", "latin america & caribbean (excluding high income)",
    "latin america & the caribbean (ida & ibrd countries)",
    "middle east & north africa", "middle east & north africa (excluding high income)",
    "middle east & north africa (ida & ibrd countries)",
    "middle east, north africa, afghanistan & pakistan",
    "north america",
    "south asia", "south asia (ida & ibrd)",
    "sub-saharan africa", "sub-saharan africa (excluding high income)",
    "sub-saharan africa (ida & ibrd countries)",
    "arab world",
    "central europe and the baltics",
    "caribbean small states", "pacific island small states",
    "other small states", "small states",
    "post-demographic dividend", "pre-demographic dividend",
    "early-demographic dividend", "late-demographic dividend",
    "oecd members",
    "channel islands", "not classified",
}

def is_aggregate(name_lower: str) -> bool:
    return name_lower in AGGREGATES

def to_iso3(names, source="?"):
    s = pd.Series(names).astype(str).str.strip()
    sl = s.str.lower()
    out = pd.Series([None]*len(s), index=s.index, dtype="object")

    # Drop aggregates
    out.loc[sl.apply(is_aggregate)] = None

    # Manual overrides
    mask = out.isna() & sl.isin(MANUAL_ISO3.keys())
    out.loc[mask] = sl.loc[mask].map(MANUAL_ISO3)

    # Everything else through country_converter
    rem = out.isna() & ~sl.apply(is_aggregate)
    if rem.any():
        r = _CC.convert(names=s.loc[rem].tolist(), to="ISO3", not_found=None)
        if isinstance(r, str):
            r = [r]
        # CRITICAL: coerce list returns (ambiguous matches) and "not found" to None
        r = [None if (isinstance(x, list) or x == "not found") else x for x in r]
        out.loc[rem] = r

    return out

print("Country code mapping ready (with list-coercion fix).")

Country code mapping ready (with list-coercion fix).


## 3. World Bank WDI — macro controls (automated)

In [4]:
import wbdata
from datetime import datetime

WDI_INDICATORS = {
    "NY.GDP.PCAP.KD.ZG": "gdp_pc_growth",
    "NY.GDP.PCAP.PP.KD": "gdp_pc_ppp",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "FP.CPI.TOTL.ZG": "inflation",
    "NE.GDI.TOTL.ZS": "investment_pct_gdp",
    "SP.POP.GROW": "pop_growth",
    "FS.AST.PRVT.GD.ZS": "credit_to_private_pct_gdp",
    "SE.SEC.ENRR": "secondary_enrollment",
    "SE.TER.ENRR": "tertiary_enrollment",
}

log.info("Downloading WDI ({} indicators)...".format(len(WDI_INDICATORS)))
wdi = wbdata.get_dataframe(WDI_INDICATORS,
    date=(datetime(YEAR_MIN,1,1), datetime(YEAR_MAX,12,31))).reset_index()
wdi = wdi.rename(columns={"country":"country_name", "date":"year"})
wdi["year"] = wdi["year"].astype(str).str[:4].astype(int)
wdi["iso3"] = to_iso3(wdi["country_name"], "WDI")
wdi = wdi.dropna(subset=["iso3"]).drop(columns="country_name")
wdi = wdi[["iso3","year"] + list(WDI_INDICATORS.values())].sort_values(["iso3","year"])
wdi.to_csv(RAW/"wdi_controls.csv", index=False)
log.info(f"WDI saved: {len(wdi):,} rows, {wdi['iso3'].nunique()} countries")
wdi.head()

,iso3,year,gdp_pc_growth,gdp_pc_ppp,trade_pct_gdp,inflation,investment_pct_gdp,pop_growth,credit_to_private_pct_gdp,secondary_enrollment,tertiary_enrollment
884,ABW,2010,-2.956953,33951.624618,135.464655,2.078141,28.283180,0.230041,58.800657,98.220741,36.062531
883,ABW,2011,2.610525,34837.940287,154.437787,4.316297,27.416637,0.736689,56.250434,96.633919,37.864811
882,ABW,2012,-2.484648,33972.340108,156.645197,0.627472,23.522507,1.469782,58.685903,108.976189,38.864182
881,ABW,2013,4.855258,35621.784855,155.731904,-2.372065,23.825860,1.492031,59.163685,NaN,17.125870
880,ABW,2014,0.351128,35746.863068,156.491053,0.421441,23.111441,1.065512,60.240539,NaN,17.751900


## 4. World Bank income classifications (automated)

Only the columns the panel needs are saved: `iso3`, `income_group`, `mi_status`, `hi_status`.

In [5]:
log.info("Downloading WB country classifications...")
r = requests.get("https://api.worldbank.org/v2/country?format=json&per_page=400", timeout=60)
data = r.json()
countries = data[1] if isinstance(data, list) and len(data) > 1 else []

inc = pd.DataFrame([{
    "iso3": c["id"],
    "income_group_label": c.get("incomeLevel", {}).get("value", None),
    "region": c.get("region", {}).get("value", None),
} for c in countries])
inc = inc[~inc["region"].isin(["Aggregates", None])]
inc["income_group"] = inc["income_group_label"].map({
    "Low income": "L", "Lower middle income": "LM",
    "Upper middle income": "UM", "High income": "H"})
inc["mi_status"] = inc["income_group"].isin(["LM","UM"]).astype(int)
inc["hi_status"] = (inc["income_group"]=="H").astype(int)

# Keep ONLY what the panel needs — names/labels/regions stay out of the merge
inc = inc[["iso3", "income_group", "mi_status", "hi_status"]].dropna(subset=["income_group"])
inc = inc.drop_duplicates(subset=["iso3"])
inc.to_csv(RAW/"income_groups_current.csv", index=False)
log.info(f"Income groups: {len(inc)} countries (current classification only — "
         f"historical year-by-year comes from CLASS.xlsx in Section 7f)")
inc.head()


,iso3,income_group,mi_status,hi_status
0,ABW,H,0,1
2,AFG,L,0,0
5,AGO,LM,1,0
6,ALB,UM,1,0
7,AND,H,0,1


## 5. Penn World Tables 10.01 (direct HTTP)

Kept columns: `iso3`, `year`, `human_capital_pwt` only — `rgdpna/pop/emp` duplicate what WDI already provides and were bloating the panel.

In [6]:
log.info("Downloading Penn World Tables 10.01...")
PWT_URL = "https://dataverse.nl/api/access/datafile/354095"  # PWT 10.01 xlsx
try:
    r = requests.get(PWT_URL, timeout=180,
                     headers={"User-Agent": "academic-research/1.0"})
    r.raise_for_status()
    raw = pd.read_excel(BytesIO(r.content), sheet_name="Data")
    pwt = (raw[["countrycode", "year", "hc"]]
           .rename(columns={"countrycode": "iso3", "hc": "human_capital_pwt"}))
    pwt = pwt[(pwt["year"] >= YEAR_MIN) & (pwt["year"] <= YEAR_MAX)]
    pwt = pwt.dropna(subset=["human_capital_pwt"])
    pwt.to_csv(RAW/"pwt.csv", index=False)
    log.info(f"PWT saved: {len(pwt):,} rows, {pwt['iso3'].nunique()} countries")
except Exception as e:
    log.error(f"PWT download failed: {e}")
    print("Manual fallback: https://www.rug.nl/ggdc/productivity/pwt/")


## 6. Global Findex — all waves, country totals, F-components only

The 2025 Findex release bundles **all waves** (2011/2014/2017/2021/2024) in one
file, but it has hundreds of columns and one row per country × wave ×
demographic subgroup. That was the main "excess variables / wrong dimensions"
problem. This cell:

1. filters to country-level totals (`group == "all"`),
2. keeps only `iso3`, `year` and the four F-index components (+ `account_t_d`),
3. saves a slim `findex_components.csv` that merges cleanly on `(iso3, year)`.

The slim file is what Section 8 merges. (Set `SAVE_FULL_RAW = True` only if you
also want the giant raw file on disk for reference.)

In [7]:
FINDEX_URL = ("https://thedocs.worldbank.org/en/doc/"
    "be6615202d1f08a25855c8ac2d615122-0050012025/related/GlobalFindexDatabase2025.csv")

# The four F-index components (+ account ownership as a sanity/extra component)
F_COMPONENTS = ["account_t_d", "fin45_t_d", "fin11_t_d", "fin22_t_d", "fin10_t_d"]
SAVE_FULL_RAW = False

log.info("Downloading Findex (2025 release, contains all waves)...")
try:
    r = requests.get(FINDEX_URL, timeout=300,
                     headers={"User-Agent": "academic-research/1.0"})
    r.raise_for_status()
    fx = pd.read_csv(BytesIO(r.content), low_memory=False)
    fx.columns = [str(c).strip().lower() for c in fx.columns]
    log.info(f"Full Findex file: {len(fx):,} rows, {len(fx.columns)} cols")

    if SAVE_FULL_RAW:
        fx.to_csv(RAW/"findex/findex_all_waves_raw.csv", index=False)

    # --- locate key columns defensively ---
    year_col = next((c for c in fx.columns if c == "year"), None)
    code_col = next((c for c in fx.columns
                     if c in ("codewb", "code", "iso3", "countrycode", "economycode")), None)
    name_col = next((c for c in fx.columns
                     if "country" in c or "economy" in c), None)
    if year_col is None:
        raise RuntimeError(f"No year column. Columns: {list(fx.columns)[:25]}")

    # --- keep country-level TOTALS only (drop demographic subgroups) ---
    group_cols = [c for c in ("group", "group2") if c in fx.columns]
    for gc in group_cols:
        vals = fx[gc].astype(str).str.strip().str.lower()
        if vals.isin(["all"]).any():
            fx = fx[vals == "all"]
    log.info(f"After filtering to country totals: {len(fx):,} rows")

    # --- iso3 ---
    if code_col is not None:
        fx["iso3"] = fx[code_col].astype(str).str.strip().str.upper()
        fx = fx[fx["iso3"].str.fullmatch(r"[A-Z]{3}")]
    elif name_col is not None:
        fx["iso3"] = to_iso3(fx[name_col], source="Findex")
        fx = fx.dropna(subset=["iso3"])
    else:
        raise RuntimeError("No country code or name column found in Findex file.")

    fx["year"] = pd.to_numeric(fx[year_col], errors="coerce").astype("Int64")
    fx = fx.dropna(subset=["year"]); fx["year"] = fx["year"].astype(int)

    have = [c for c in F_COMPONENTS if c in fx.columns]
    missing = [c for c in F_COMPONENTS if c not in fx.columns]
    if missing:
        log.warning(f"F-components not in file (older-wave naming?): {missing}")
    if not have:
        raise RuntimeError("None of the F-components found — inspect fx.columns")

    findex = fx[["iso3", "year"] + have].copy()
    for c in have:
        findex[c] = pd.to_numeric(findex[c], errors="coerce")
    findex = findex.rename(columns={c: f"findex_{c}" for c in have})
    # one row per country-wave
    findex = (findex.groupby(["iso3", "year"], as_index=False).mean(numeric_only=True))
    findex = findex[(findex["year"] >= 2011) & (findex["year"] <= YEAR_MAX)]

    findex.to_csv(RAW/"findex_components.csv", index=False)
    log.info(f"Findex slim file saved: {len(findex):,} country-wave rows, "
             f"{findex['iso3'].nunique()} countries, "
             f"waves {sorted(findex['year'].unique())}")
    print("\nCoverage (non-null) per wave:")
    print(findex.groupby("year").count())
except Exception as e:
    log.error(f"Findex download failed: {e}")
    print("Manual fallback: https://www.worldbank.org/en/publication/globalfindex/download-data")



Coverage (non-null) per wave:
      iso3  findex_account_t_d
year                          
2011   156                 156
2014   154                 154
2017   158                 158
2021   135                 135
2022    16                  16
2024   152                 152


## 7. OpenAlex — AI publications by country-year (Stanford AI Index fallback)

This is the FALLBACK for the Activity pillar of the AI index. When you receive
the Stanford AI Index official panel, replace the output of this section.

**SWAP-IN POINT:** if Stanford AI Index data arrives, save it as
`/content/data/raw/stanford_ai_index.csv` with columns
`iso3, year, ai_investment, ai_patents_count, ai_talent_share`. The
downstream G-index construction will pick it up automatically.

In [8]:
AI_CONCEPT = "C154945302"  # Artificial intelligence
years_to_query = range(YEAR_MIN, YEAR_MAX + 1)   # full panel range, not just 2017+

log.info(f"Querying OpenAlex for AI publications, years {min(years_to_query)}-{max(years_to_query)}")
all_rows = []
for y in years_to_query:
    url = (f"https://api.openalex.org/works"
           f"?filter=concepts.id:{AI_CONCEPT},publication_year:{y},type:article"
           f"&group_by=authorships.institutions.country_code&per-page=200"
           f"&mailto=research@example.org")
    ok = False
    for attempt in range(3):                      # retry transient failures
        try:
            resp = requests.get(url, timeout=60,
                                headers={"User-Agent": "academic-research/1.0"})
            resp.raise_for_status()
            d = resp.json()
            groups = d.get("group_by", [])
            for g in groups:
                key = str(g.get("key", "")).split("/")[-1]   # tolerate URL-style keys
                if key and key.lower() != "unknown":
                    all_rows.append({"country_iso2": key.lower(),
                                     "year": y, "ai_pubs_count": g["count"]})
            log.info(f"  year {y}: {len(groups)} countries")
            ok = True
            break
        except Exception as e:
            log.warning(f"  year {y} attempt {attempt+1}: {e}")
            time.sleep(2 * (attempt + 1))
    if not ok:
        log.error(f"  year {y}: FAILED after retries — this year will be missing")
    time.sleep(0.3)  # be polite to free API

import pycountry
def iso2_to_iso3(c):
    try: return pycountry.countries.get(alpha_2=c.upper()).alpha_3
    except Exception: return None

openalex = pd.DataFrame(all_rows)
if len(openalex) > 0:
    openalex["iso3"] = openalex["country_iso2"].apply(iso2_to_iso3)
    openalex = openalex.dropna(subset=["iso3"])[["iso3", "year", "ai_pubs_count"]]
    openalex.to_csv(RAW/"openalex_ai_pubs.csv", index=False)
    log.info(f"OpenAlex saved: {len(openalex):,} country-year rows, "
             f"years {openalex['year'].min()}-{openalex['year'].max()}")
    print(openalex.head())
else:
    # IMPORTANT: do NOT write an empty file — Section 8 treats missing as "skip"
    # but an empty file would silently produce all-NaN columns in the panel.
    log.error("OpenAlex returned no data at all — openalex_ai_pubs.csv NOT written. "
              "Re-run this cell (the API may have been temporarily down).")


  iso3  year  ai_pubs_count
0  CHN  2010         102897
1  USA  2010          60740
2  GBR  2010          15978
3  JPN  2010          15822
4  DEU  2010          13597


## 7b. ILO WP 140 — GenAI occupational exposure (income-group aggregate)

The ILO Working Paper 140 publishes the country-level GenAI exposure shares only
as aggregates by **income group** and **region** (not by individual country). The
numbers below are transcribed from the interactive chart at
https://www.ilo.org/resource/article/how-might-generative-ai-impact-different-occupations
(Datawrapper id `noouS`, also reproduced in Table 8 of WP 140).

We merge these onto the panel via the WB income group. This means every High-income
country gets the same value (6.2% in Gradient 4), every UM gets 3.6%, etc. This is
a coarse but valid measure for **robustness checks** against a primary country-level
AI-exposure measure (e.g. IMF Cazzaniga 2024). It is collinear with income-group
fixed effects, so use it in specs without those FEs.


In [9]:
# ILO WP 140 income-group aggregates (Gradient 4 = highest GenAI exposure share, %)
# Source: ILO interactive chart, accessed June 2026.
ilo_rho = pd.DataFrame({
    "income_group":     ["H",  "UM", "LM", "L"],
    "rho_gradient4":    [6.2,  3.6,  1.9,  0.5],
    "rho_gradient3":    [10.9, 3.4,  1.9,  0.6],
    # TODO: fill in once you have the remaining columns visible
    # "rho_gradient2":  [...],
    # "rho_gradient1":  [...],
    # "rho_minimal":    [...],
    # "rho_not_exposed":[...],
})
ilo_rho["rho_exposed_top2"] = ilo_rho["rho_gradient4"] + ilo_rho["rho_gradient3"]
ilo_rho.to_csv(RAW/"ilo_rho_by_income.csv", index=False)
log.info(f"ILO WP140 income-group aggregates saved: {len(ilo_rho)} groups")
ilo_rho


,income_group,rho_gradient4,rho_gradient3,rho_exposed_top2
0,H,6.2,10.9,17.1
1,UM,3.6,3.4,7.0
2,LM,1.9,1.9,3.8
3,L,0.5,0.6,1.1


## 7c. Oxford GARI 2024 — upload first, PDF scrape fallback

Run this cell and it will **pop a file-upload picker**: drop in either

- a spreadsheet you built from the Oxford Insights rankings table
  (`GARI_2024.xlsx`/`.csv` with Rank / Country / Total score columns), **or**
- the official report **PDF** itself — the cell extracts the rankings table
  from it automatically.

If files already sit in `/content/data/raw/oxford_gari/`, those are used
without prompting. If you press **Cancel** (or upload nothing), the cell
downloads the official 2024 report PDF from Oxford Insights and scrapes the
table itself.

Output: `gari_2024.csv` with exactly `iso3, year, gari_total` — validated
(scores within 0–100, deduplicated, ≥100 countries required for any scraped
table before it's trusted).

In [10]:
import re

GARI_DIR = RAW/"oxford_gari"
GARI_DIR.mkdir(parents=True, exist_ok=True)
GARI_PDF_URLS = [
    "https://oxfordinsights.com/wp-content/uploads/2024/12/2024-Government-AI-Readiness-Index-2.pdf",
    "https://oxfordinsights.com/wp-content/uploads/2024/12/2024-Government-AI-Readiness-Index.pdf",
]

# ---------- parse a spreadsheet (manual export) ----------
def parse_gari_table_file(fpath):
    df = (pd.read_excel(fpath) if fpath.suffix.lower() in (".xlsx", ".xls")
          else pd.read_csv(fpath))
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    score_col   = next((c for c in df.columns if "total" in c or "score" in c), None)
    country_col = next((c for c in df.columns if "country" in c or "economy" in c), None)
    if score_col is None or country_col is None:
        raise ValueError(f"need country + total/score columns, got {list(df.columns)}")
    out = df[[country_col, score_col]].rename(
        columns={country_col: "country", score_col: "gari_total"})
    out["gari_total"] = pd.to_numeric(out["gari_total"], errors="coerce")
    return out.dropna(subset=["gari_total"])

# ---------- extract the rankings table from the report PDF ----------
def parse_gari_pdf(pdf_path, min_rows=100):
    try:
        import pdfplumber
    except ImportError:
        !pip install -q pdfplumber
        import pdfplumber
    # Rows look like: "12  United Kingdom  78.88 ..."
    ROW_RE = re.compile(r"^\s*(\d{1,3})\s+([A-Za-z][A-Za-z'’.,()&\- ]+?)\s+(\d{1,3}\.\d{1,2})\b")
    rows = {}
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for line in (page.extract_text() or "").split("\n"):
                m = ROW_RE.match(line)
                if not m:
                    continue
                rank, country, score = int(m.group(1)), m.group(2).strip(), float(m.group(3))
                if 1 <= rank <= 200 and 0 < score <= 100 and country not in rows:
                    rows[country] = {"country": country, "gari_total": score}
    if len(rows) < min_rows:
        raise ValueError(f"PDF extraction found only {len(rows)} plausible rows — not trusting it")
    return pd.DataFrame(rows.values())

# ================= main flow: upload first, scrape as fallback =================
gari = None

# 1) files already in the folder?
def _gari_files():
    return sorted(list(GARI_DIR.glob("*.xlsx")) + list(GARI_DIR.glob("*.xls"))
                  + list(GARI_DIR.glob("*.csv")) + list(GARI_DIR.glob("*.pdf")))
files_found = _gari_files()

# 2) nothing there -> pop the Colab upload picker
if not files_found:
    try:
        from google.colab import files as colab_files
        print("Upload your GARI file now (xlsx/csv rankings table, or the report PDF) — "
              "or press Cancel to auto-download and scrape the official PDF.")
        uploaded = colab_files.upload()
        for name, content in uploaded.items():
            dst = GARI_DIR/Path(name).name
            dst.write_bytes(content)
            log.info(f"  saved upload: {dst}")
        files_found = _gari_files()
    except ImportError:
        pass  # not in Colab
    except Exception as e:
        log.warning(f"  upload skipped: {e}")

# 3) parse what we have: spreadsheets take priority over PDFs
tables = [f for f in files_found if f.suffix.lower() != ".pdf"]
pdfs   = [f for f in files_found if f.suffix.lower() == ".pdf"]
for fpath in tables:
    try:
        gari = parse_gari_table_file(fpath)
        log.info(f"GARI parsed from spreadsheet {fpath.name}: {len(gari)} rows")
        break
    except Exception as e:
        log.warning(f"  could not parse {fpath.name}: {e}")
if gari is None:
    for fpath in pdfs:
        try:
            gari = parse_gari_pdf(fpath)
            log.info(f"GARI extracted from uploaded PDF {fpath.name}: {len(gari)} rows")
            break
        except Exception as e:
            log.warning(f"  could not extract from {fpath.name}: {e}")

# 4) still nothing -> download the official report PDF and scrape it
if gari is None:
    pdf_path = GARI_DIR/"GARI_2024_report.pdf"
    if not pdf_path.exists():
        for url in GARI_PDF_URLS:
            try:
                r = requests.get(url, timeout=180, headers={"User-Agent": "academic/1.0"})
                r.raise_for_status()
                if len(r.content) < 100_000:
                    raise RuntimeError("file too small to be the report")
                pdf_path.write_bytes(r.content)
                log.info(f"GARI report PDF downloaded: {pdf_path.stat().st_size:,} bytes")
                break
            except Exception as e:
                log.warning(f"  PDF download failed from {url}: {e}")
    if pdf_path.exists():
        try:
            gari = parse_gari_pdf(pdf_path)
            log.info(f"GARI extracted from official PDF: {len(gari)} rows")
        except Exception as e:
            log.error(f"  {e}")

# 5) clean, validate, cut to exactly what the panel needs, write
if gari is not None and len(gari):
    gari["country"] = gari["country"].astype(str).str.strip()
    gari["iso3"] = to_iso3(gari["country"], source="GARI")
    gari = gari.dropna(subset=["iso3"]).copy()
    gari["gari_total"] = pd.to_numeric(gari["gari_total"], errors="coerce")
    bad = ((gari["gari_total"] <= 0) | (gari["gari_total"] > 100)).sum()
    if bad:
        log.warning(f"  {bad} rows with implausible scores dropped")
        gari = gari[(gari["gari_total"] > 0) & (gari["gari_total"] <= 100)]
    gari = gari.dropna(subset=["gari_total"]).drop_duplicates(subset=["iso3"])
    gari["year"] = 2024  # informational; Section 8 merges on iso3
    gari = gari[["iso3", "year", "gari_total"]].sort_values("iso3").reset_index(drop=True)
    gari.to_csv(RAW/"gari_2024.csv", index=False)
    log.info(f"GARI saved: {len(gari)} countries "
             f"(scores {gari['gari_total'].min():.1f}-{gari['gari_total'].max():.1f})")
    print(gari.head())
else:
    log.error(f"GARI could not be obtained from uploads OR the official PDF. "
              f"Drop GARI_2024.xlsx (Rank/Country/Total score) or the report PDF "
              f"into {GARI_DIR}, then re-run this cell.")


Upload your GARI file now (xlsx/csv rankings table, or the report PDF) — or press Cancel to auto-download and scrape the official PDF.


Saving GARI_2024.xlsx to GARI_2024.xlsx
  iso3  year  gari_total
0  AFG  2024   16.924518
1  AGO  2024   26.907980
2  ALB  2024   45.466228
3  AND  2024   54.436037
4  ARE  2024   75.658046


## 7d. Worldwide Governance Indicators (WGI)

The cell below tries to **auto-download** `wgidataset.xlsx` first; manual upload
to `/content/data/raw/wgi/wgidataset.xlsx` is only a fallback. The parser keeps
only the six `estimate` series, reshaped to one column per indicator.

In [11]:
WGI_PATH = RAW/"wgi/wgidataset.xlsx"
if not WGI_PATH.exists():
    try:
        url = "https://www.worldbank.org/content/dam/sites/govindicators/doc/wgidataset.xlsx"
        r = requests.get(url, timeout=120, headers={"User-Agent": "academic/1.0"})
        r.raise_for_status()
        WGI_PATH.write_bytes(r.content)
        log.info(f"WGI auto-downloaded: {WGI_PATH.stat().st_size:,} bytes")
    except Exception as e:
        log.warning(f"WGI auto-download failed ({e}). Upload wgidataset.xlsx to {WGI_PATH} manually.")
else:
    log.info(f"WGI already present at {WGI_PATH}")


In [12]:
WGI_PATH = RAW/"wgi/wgidataset.xlsx"

SHEET_TO_CODE = {
    "VoiceandAccountability":         "va",
    "Political StabilityNoViolence":  "pv",
    "GovernmentEffectiveness":        "ge",
    "RegulatoryQuality":              "rq",
    "RuleofLaw":                      "rl",
    "ControlofCorruption":            "cc",
}

def parse_wgi_sheet(path, sheet, ind_code):
    """WGI per-indicator sheet:
       Row N:   year labels (merged, repeated across sub-cols)
       Row N+1: 'Country/Territory | Code | Estimate | StdErr | NumSrc | Rank | Lower | Upper'  (repeated per year)
       Row N+2+: data
    """
    raw = pd.read_excel(path, sheet_name=sheet, header=None)

    # Find the REAL header row — must contain Country/Territory AND Code AND Estimate AND StdErr
    sub_row = None
    for i in range(min(25, len(raw))):
        cells = [str(v).strip().lower() for v in raw.iloc[i].tolist()]
        cell_set = set(cells)
        if ("estimate" in cell_set and "stderr" in cell_set
                and any("country" in c for c in cells)
                and "code" in cell_set):
            sub_row = i
            break
    if sub_row is None:
        log.warning(f"  {sheet}: real header row not found; skipping")
        return None

    years = raw.iloc[sub_row - 1].ffill()
    subs  = raw.iloc[sub_row]

    new_cols = []
    for y, s in zip(years, subs):
        try:
            yi = int(float(y))
        except (ValueError, TypeError):
            yi = None
        new_cols.append((yi, str(s).strip().lower() if pd.notna(s) else ""))

    data = raw.iloc[sub_row + 1:].copy()
    data.columns = new_cols

    # Locate country and code columns by their labels (year is None for these)
    country_col = next((c for c in new_cols if c[1].startswith("country")), None)
    code_col    = next((c for c in new_cols if c[1] == "code"), None)
    if country_col is None or code_col is None:
        log.warning(f"  {sheet}: country/code columns not found")
        return None

    # Estimate columns with valid years
    est_cols = [(y, s) for (y, s) in new_cols if s == "estimate" and y is not None]
    if not est_cols:
        log.warning(f"  {sheet}: no estimate columns parsed")
        return None

    pieces = []
    for col in est_cols:
        y = col[0]
        sub = data[[country_col, code_col, col]].copy()
        sub.columns = ["country", "iso3_wgi", "estimate"]
        sub["year"] = y
        sub["ind_code"] = ind_code
        pieces.append(sub)

    out = pd.concat(pieces, ignore_index=True)
    out["estimate"] = pd.to_numeric(out["estimate"], errors="coerce")
    out["country"]  = out["country"].astype(str).str.strip()
    out["iso3_wgi"] = out["iso3_wgi"].astype(str).str.strip().str.upper()
    out = out.dropna(subset=["country", "estimate"])
    out = out[out["country"].str.len() > 0]
    out = out[~out["country"].str.lower().isin({"nan", "none"})]
    return out

if not WGI_PATH.exists():
    log.warning(f"WGI file not found at {WGI_PATH}")
else:
    log.info(f"Reading WGI from {WGI_PATH}")
    xl = pd.ExcelFile(WGI_PATH)
    log.info(f"  sheets: {xl.sheet_names}")

    pieces = []
    for sheet, code in SHEET_TO_CODE.items():
        if sheet not in xl.sheet_names:
            log.warning(f"  expected sheet '{sheet}' missing; skipping {code}")
            continue
        df = parse_wgi_sheet(WGI_PATH, sheet, code)
        if df is not None:
            log.info(f"  {sheet} → {code}: {len(df):,} country-years")
            pieces.append(df)

    if not pieces:
        raise RuntimeError("Could not parse any WGI sheets.")

    wgi_long = pd.concat(pieces, ignore_index=True)

    # WGI's embedded Code column is mostly ISO3 but has a few legacy codes
    WGI_LEGACY_FIX = {"ADO": "AND", "ROM": "ROU", "ZAR": "COD", "TMP": "TLS",
                      "WBG": "PSE", "KSV": "XKX"}
    wgi_long["iso3"] = wgi_long["iso3_wgi"].map(WGI_LEGACY_FIX).fillna(wgi_long["iso3_wgi"])
    # Anything that isn't a clean 3-letter code → resolve from country name
    bad_mask = ~wgi_long["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
    if bad_mask.any():
        wgi_long.loc[bad_mask, "iso3"] = to_iso3(
            wgi_long.loc[bad_mask, "country"], source="WGI"
        )
    wgi_long = wgi_long.dropna(subset=["iso3"]).copy()
    wgi_long["year"] = pd.to_numeric(wgi_long["year"], errors="coerce").astype("Int64")
    wgi_long = wgi_long[(wgi_long["year"] >= YEAR_MIN) & (wgi_long["year"] <= YEAR_MAX)]
    wgi_long["year"] = wgi_long["year"].astype(int)

    wgi_wide = (wgi_long.pivot_table(index=["iso3", "year"], columns="ind_code",
                                     values="estimate", aggfunc="mean")
                .reset_index())

    wgi_wide.columns.name = None
    wgi_wide = wgi_wide.rename(columns={c: f"wgi_{c}" for c in wgi_wide.columns
                                         if c not in ("iso3", "year")})
    wgi_wide.to_csv(RAW/"wgi.csv", index=False)
    log.info(f"WGI saved: {len(wgi_wide):,} rows, {wgi_wide['iso3'].nunique()} countries, "
             f"indicators: {[c for c in wgi_wide.columns if c.startswith('wgi_')]}")
    print(wgi_wide.head())

/tmp/ipykernel_3912/1036069060.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  years = raw.iloc[sub_row - 1].ffill()
/tmp/ipykernel_3912/1036069060.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  years = raw.iloc[sub_row - 1].ffill()
/tmp/ipykernel_3912/1036069060.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  years = raw.

  iso3  year    wgi_cc    wgi_ge    wgi_pv    wgi_rl    wgi_rq    wgi_va
0  ABW  2010  1.121296  1.246453  1.100847  1.416114  1.344838  1.281884
1  ABW  2011  1.102947  1.236430  1.285738  1.322458  1.317277  1.273588
2  ABW  2012  1.095236  1.247588  1.263882  1.309788  1.402131  1.284650
3  ABW  2013  1.119407  1.221326  1.312226  1.292505  1.420880  1.278059
4  ABW  2014  1.014232  0.887890  1.164568  1.036310  1.245361  1.276850


/tmp/ipykernel_3912/1036069060.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  years = raw.iloc[sub_row - 1].ffill()


## 7e. IMF AI Preparedness Index (AIPI) — upload first, API fallback

Run this cell and it will **pop a file-upload picker**: drop in your export(s)
from https://www.imf.org/external/datamapper/datasets/AIPI (Export → Excel or
CSV — either one combined file, or one file per indicator). The cell parses
whatever you upload.

- If files are already sitting in `/content/data/raw/imf_aipi/`, those are used
  without prompting.
- If you press **Cancel** on the picker (or upload nothing), the cell falls
  back to **scraping the IMF DataMapper API automatically** instead.

Either way the output is `imf_aipi.csv` (one row per country, `aipi_*`
columns), which Section 8 merges as a snapshot on `iso3`.

In [15]:
AIPI_DIR = RAW/"imf_aipi"
AIPI_DIR.mkdir(parents=True, exist_ok=True)
UA = {"User-Agent": "academic-research/1.0"}
API = "https://www.imf.org/external/datamapper/api/v1"

import pycountry
VALID_ISO3 = {c.alpha_3 for c in pycountry.countries} | {"XKX"}  # Kosovo

def aipi_short_label(text):
    ll = str(text).lower()
    if "digital" in ll:                      return "digital_infra"
    if "human" in ll:                        return "human_capital"
    if "innovation" in ll or "innocation" in ll: return "innovation"
    if "regulation" in ll or "ethic" in ll:  return "regulation"
    if "preparedness" in ll or "ai_pi" in ll or "aipi" in ll: return "total"
    return None

# ---------- parse one uploaded file -> frame keyed by country ----------
def _year_like(v):
    try:
        y = int(float(str(v).strip()))
        return 1980 <= y <= 2035 and float(str(v).strip()) == y
    except (ValueError, TypeError):
        return False

def _num_like(v):
    try:
        float(str(v).strip()); return True
    except (ValueError, TypeError):
        return False

def _read_grid(fpath):
    """Read any csv/xlsx into a raw header-less grid, sniffing the delimiter."""
    if fpath.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(fpath, header=None)
    import csv as _csv
    raw = fpath.read_text(encoding="utf-8-sig", errors="replace")
    lines = [l for l in raw.splitlines() if l.strip()]
    best_delim, best_cols = ",", 1
    for d in [",", ";", "\t", "|"]:
        try:
            counts = [len(next(_csv.reader([l], delimiter=d))) for l in lines[:30]]
        except Exception:
            continue
        if counts and max(counts) > best_cols:
            best_cols, best_delim = max(counts), d
    rows = [next(_csv.reader([l], delimiter=best_delim)) for l in lines]
    width = max(len(r) for r in rows)
    rows = [r + [None]*(width - len(r)) for r in rows]
    return pd.DataFrame(rows)

def parse_aipi_file(fpath):
    """Handles every IMF DataMapper export layout:
       A) per-indicator, headerless:   title line, then  Country,0.52  rows
       B) per-indicator, year header:  country,2023      then data rows
       C) combined: Country, <indicator name>, <indicator name>, ...
       Returns frame with 'country' + one or more aipi_* columns."""
    grid = _read_grid(fpath)
    grid = grid.dropna(how="all").reset_index(drop=True)

    # find the first row with >= 2 non-empty cells
    first_wide = None
    for i in range(min(15, len(grid))):
        vals = [v for v in grid.iloc[i] if pd.notna(v) and str(v).strip() != ""]
        if len(vals) >= 2:
            first_wide = i
            break
    if first_wide is None:
        raise ValueError(f"{fpath.name}: never found a row with >=2 columns; "
                         f"first lines: {grid.head(3).values.tolist()}")

    row = [str(v).strip() if pd.notna(v) else "" for v in grid.iloc[first_wide]]
    tail = [v for v in row[1:] if v != ""]

    if all(_year_like(v) for v in tail):
        kind = "year_header"           # e.g.  country | 2023
    elif all(_num_like(v) for v in tail):
        kind = "headerless"            # e.g.  Afghanistan | 0.25  (data already!)
    else:
        kind = "named_header"          # e.g.  Country | Digital Infrastructure | ...

    file_lbl = aipi_short_label(fpath.name)

    if kind == "headerless":
        data = grid.iloc[first_wide:].copy()
        data.columns = range(data.shape[1])
        out = pd.DataFrame({
            "country": data[0].astype(str).str.strip(),
            f"aipi_{file_lbl or 'total'}": pd.to_numeric(data[1], errors="coerce"),
        })
        return out.dropna(subset=[out.columns[1]])

    header = row
    data = grid.iloc[first_wide + 1:].copy()
    data.columns = [h if h else f"col{j}" for j, h in enumerate(header)]

    country_col = next((c for c in data.columns
                        if "country" in str(c).lower() or str(c).lower() == "economy"),
                       data.columns[0])
    data = data.rename(columns={country_col: "country"})
    data["country"] = data["country"].astype(str).str.strip()
    data = data[data["country"].str.len() > 0]
    data = data[~data["country"].str.lower().isin({"nan", "none"})]

    if kind == "year_header":
        year_cols = [c for c in data.columns if _year_like(c)]
        for c in year_cols:
            data[c] = pd.to_numeric(data[c], errors="coerce")
        recent_first = sorted(year_cols, key=lambda c: int(float(str(c))), reverse=True)
        lbl = file_lbl or aipi_short_label(header[0]) or "total"
        out = data[["country"]].copy()
        out[f"aipi_{lbl}"] = data[recent_first].bfill(axis=1).iloc[:, 0]
        return out.dropna(subset=[f"aipi_{lbl}"])

    # named_header (combined file): every plausible numeric indicator column
    keep = {}
    for c in data.columns:
        if c == "country":
            continue
        lbl = aipi_short_label(c)
        if lbl: # Only process if a valid label is found
            vals = pd.to_numeric(data[c], errors="coerce")
            if vals.notna().sum() < 5: # Reduced threshold to 5 from 20
                continue
            keep[f"aipi_{lbl}"] = vals
    if not keep:
        raise ValueError(f"{fpath.name}: no usable indicator columns; "
                         f"header was {header}")
    out = pd.concat([data[["country"]]] + [s.rename(k) for k, s in keep.items()], axis=1)
    return out

# ---------- API scrape fallback ----------
BROWSER_UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
              "Accept": "application/json"}

def _get_json(url):
    for hdrs in (BROWSER_UA, UA):
        try:
            r = requests.get(url, timeout=60, headers=hdrs)
            r.raise_for_status()
            return r.json()
        except ValueError:
            log.warning(f"  non-JSON response from {url} "
                        f"(status {r.status_code}, starts: {r.text[:80]!r})")
        except Exception as e:
            log.warning(f"  {url}: {e}")
    return None

def scrape_aipi_api():
    j = _get_json(f"{API}/indicators")
    if j is None:
        log.error("  API indicator discovery failed (IMF may be blocking this "
                  "Colab IP) — rely on the uploaded files instead.")
        return None
    inds = j.get("indicators", {})
    codes = {k: v for k, v in inds.items()
             if str((v or {}).get("dataset", "")).upper() == "AIPI"} or \
            {"AI_PI": {"label": "AI Preparedness Index"},
             "HCLMP": {"label": "Human Capital and Labor Market Policies"},
             "IEI":   {"label": "Innovation and Economic Integration"}}
    log.info(f"  AIPI indicators on DataMapper: "
             f"{ {k: (v or {}).get('label') for k, v in codes.items()} }")
    frames = []
    for k, v in codes.items():
        jk = _get_json(f"{API}/{k}")
        if jk is None:
            log.warning(f"  {k} failed")
            continue
        vals = jk.get("values", {}).get(k, {})
        rows = []
        for iso3, series in vals.items():
            iso3 = str(iso3).strip().upper()
            if iso3 not in VALID_ISO3 or not isinstance(series, dict):
                continue  # drops aggregates like EME, LIC, ADVEC
            for y in sorted((y for y in series if str(y).isdigit()), key=int, reverse=True):
                val = pd.to_numeric(pd.Series([series[y]]), errors="coerce").iloc[0]
                if pd.notna(val):
                    lbl = aipi_short_label((v or {}).get("label", k)) or k.lower()
                    rows.append({"iso3": iso3, f"aipi_{lbl}": float(val)})
                    break
        if rows:
            frames.append(pd.DataFrame(rows).drop_duplicates("iso3"))
            log.info(f"  {k}: {len(rows)} countries")
        time.sleep(0.3)
    if not frames:
        return None
    out = frames[0]
    for f in frames[1:]:
        out = out.merge(f, on="iso3", how="outer")
    return out

# ================= main flow: upload first, scrape as fallback =================
aipi_out = None

# 1) files already in the folder?
files_found = sorted(list(AIPI_DIR.glob("*.xlsx")) + list(AIPI_DIR.glob("*.xls"))
                     + list(AIPI_DIR.glob("*.csv")))

# 2) nothing there -> pop the Colab upload picker
if not files_found:
    try:
        from google.colab import files as colab_files
        print("Upload your IMF AIPI export(s) now (xlsx/csv) — or press Cancel "
              "to scrape the IMF API instead.")
        uploaded = colab_files.upload()           # blocks until you pick or cancel
        for name, content in uploaded.items():
            dst = AIPI_DIR/Path(name).name
            dst.write_bytes(content)
            log.info(f"  saved upload: {dst}")
        files_found = sorted(list(AIPI_DIR.glob("*.xlsx")) + list(AIPI_DIR.glob("*.xls"))
                             + list(AIPI_DIR.glob("*.csv")))
    except ImportError:
        pass  # not running in Colab — skip straight to the API
    except Exception as e:
        log.warning(f"  upload skipped: {e}")

# 3) parse uploaded/present files
if files_found:
    log.info(f"Parsing {len(files_found)} AIPI file(s): {[f.name for f in files_found]}")
    frames = []
    for fpath in files_found:
        try:
            frames.append(parse_aipi_file(fpath))
        except Exception as e:
            log.warning(f"  could not parse {fpath.name}: {e}")
    if frames:
        merged = frames[0]
        for f in frames[1:]:
            merged = merged.merge(f, on="country", how="outer")
        merged["iso3"] = to_iso3(merged["country"], source="IMF AIPI")
        merged = merged.dropna(subset=["iso3"]).drop(columns=["country"])
        aipi_out = merged.groupby("iso3", as_index=False).mean(numeric_only=True)

# 4) no usable upload -> scrape the API
if aipi_out is None or len(aipi_out) < 50:
    log.info("No usable uploaded files — scraping the IMF DataMapper API...")
    scraped = scrape_aipi_api()
    if scraped is not None:
        aipi_out = scraped

# 5) save
if aipi_out is not None and len(aipi_out):
    aipi_out["year"] = 2023   # snapshot year (informational; Section 8 merges on iso3)
    aipi_out = aipi_out.drop_duplicates("iso3").reset_index(drop=True)
    aipi_out.to_csv(RAW/"imf_aipi.csv", index=False)
    log.info(f"AIPI saved: {len(aipi_out)} countries, "
             f"{[c for c in aipi_out.columns if c.startswith('aipi_')]}")
    print(aipi_out.head())
else:
    log.error("AIPI could not be obtained from uploads OR the API. "
              "Export from https://www.imf.org/external/datamapper/datasets/AIPI "
              f"and drop the file into {AIPI_DIR}, then re-run this cell.")

ERROR:data:  API indicator discovery failed (IMF may be blocking this Colab IP) — rely on the uploaded files instead.
ERROR:data:AIPI could not be obtained from uploads OR the API. Export from https://www.imf.org/external/datamapper/datasets/AIPI and drop the file into /content/data/raw/imf_aipi, then re-run this cell.


## 7f. WB OGHIST.xlsx — historical income classifications

**Fixed:** the historical year-by-year classification is NOT in `CLASS.xlsx`
(that file only holds the current year). It lives in a separate file,
`OGHIST.xlsx` ("Country Analytical History" sheet), downloaded from the same
World Bank page. This cell auto-downloads it; manual fallback is to upload it
to `/content/data/raw/wb_income/OGHIST.xlsx`.

The sheet has ISO3 codes in the first column and one column per calendar year
with codes `L / LM / UM / H` (plus `..` for unclassified and `*` footnote
markers, which we clean). Output: `income_groups_hist.csv` with
`iso3, year, income_group_hist, mi_status_hist, hi_status_hist`.

In [ ]:
OGHIST_PATH = RAW/"wb_income/OGHIST.xlsx"
OGHIST_URLS = [
    # Official "historical classification by income" link on the WB country-groups page
    "https://ddh-openapi.worldbank.org/resources/DR0095334/download",
    "https://datacatalogfiles.worldbank.org/ddh-published/0037712/DR0090754/OGHIST.xlsx",
]
if not OGHIST_PATH.exists():
    for url in OGHIST_URLS:
        try:
            r = requests.get(url, timeout=120, headers={"User-Agent": "academic/1.0"})
            r.raise_for_status()
            if len(r.content) < 10_000:
                raise RuntimeError(f"suspiciously small file ({len(r.content)} bytes)")
            OGHIST_PATH.write_bytes(r.content)
            log.info(f"OGHIST.xlsx downloaded from {url}: {OGHIST_PATH.stat().st_size:,} bytes")
            break
        except Exception as e:
            log.warning(f"OGHIST download failed from {url}: {e}")
    else:
        log.error(f"All OGHIST downloads failed. Manually download the 'historical "
                  f"classification by income' file from "
                  f"https://datahelpdesk.worldbank.org/knowledgebase/articles/906519 "
                  f"and upload it to {OGHIST_PATH}")
else:
    log.info(f"OGHIST.xlsx already present at {OGHIST_PATH}")


In [ ]:
if not OGHIST_PATH.exists():
    log.warning(f"OGHIST.xlsx not found at {OGHIST_PATH} — skipping income history")
else:
    log.info(f"Reading OGHIST from {OGHIST_PATH}")
    xl = pd.ExcelFile(OGHIST_PATH)
    log.info(f"  sheets: {xl.sheet_names}")
    sheet = next((s for s in xl.sheet_names if "hist" in s.lower() or "analytical" in s.lower()),
                 xl.sheet_names[0])
    log.info(f"  using sheet: {sheet}")

    raw = pd.read_excel(OGHIST_PATH, sheet_name=sheet, header=None)

    # --- find the header row: the row with the most cells that parse as years ---
    def year_or_none(v):
        try:
            y = int(float(str(v).strip()))
            return y if 1985 <= y <= 2035 else None
        except (ValueError, TypeError):
            return None

    best_row, best_count = None, 0
    for i in range(min(15, len(raw))):
        cnt = sum(year_or_none(v) is not None for v in raw.iloc[i].tolist())
        if cnt > best_count:
            best_row, best_count = i, cnt
    if best_row is None or best_count < 5:
        raise RuntimeError("Could not locate a year header row in OGHIST — "
                           "inspect the sheet layout manually.")
    log.info(f"  year header row: {best_row} ({best_count} year columns)")

    col_year = {j: year_or_none(v) for j, v in enumerate(raw.iloc[best_row].tolist())}
    year_cols = [j for j, y in col_year.items() if y is not None]

    # --- data rows: first column holds 3-letter ISO codes ---
    data = raw.iloc[best_row + 1:].copy()
    iso = data.iloc[:, 0].astype(str).str.strip().str.upper()
    data = data[iso.str.fullmatch(r"[A-Z]{3}")]
    log.info(f"  {len(data)} country rows detected")

    records = []
    for _, row in data.iterrows():
        iso3 = str(row.iloc[0]).strip().upper()
        for j in year_cols:
            records.append({"iso3": iso3, "year": col_year[j], "ig_raw": row.iloc[j]})
    long = pd.DataFrame(records)

    # --- clean codes: strip footnote '*', map legacy codes, drop '..' ---
    CODE_MAP = {
        "L":"L","LIC":"L",
        "LM":"LM","LMC":"LM","LMI":"LM",
        "UM":"UM","UMC":"UM","UMI":"UM",
        "H":"H","HIC":"H","OECD":"H","NONOECD":"H",
    }
    long["ig_clean"] = (long["ig_raw"].astype(str).str.strip().str.upper()
                        .str.replace("*", "", regex=False)
                        .str.replace("-", "", regex=False)
                        .map(CODE_MAP))
    long = long.dropna(subset=["ig_clean"])
    long = long[(long["year"] >= YEAR_MIN) & (long["year"] <= YEAR_MAX)]

    out = long[["iso3","year","ig_clean"]].rename(columns={"ig_clean": "income_group_hist"})
    out["mi_status_hist"] = out["income_group_hist"].isin(["LM","UM"]).astype(int)
    out["hi_status_hist"] = (out["income_group_hist"] == "H").astype(int)
    out = out.drop_duplicates(subset=["iso3","year"]).sort_values(["iso3","year"])

    if len(out) == 0:
        log.error("OGHIST parsed to 0 rows — NOT writing income_groups_hist.csv")
    else:
        out.to_csv(RAW/"income_groups_hist.csv", index=False)
        log.info(f"Income history saved: {len(out):,} country-year rows, "
                 f"{out['iso3'].nunique()} countries, "
                 f"{out['year'].min()}-{out['year'].max()}")
        print(out.head())


## 7g. ILOSTAT — KIS share (Knowledge-Intensive Services)

Pull employment by economic activity (ISIC Rev. 4) from ILOSTAT and compute the
**KIS share** = (employment in J + K + M) / (total employment), where:
- **J** = Information and communication
- **K** = Financial and insurance activities
- **M** = Professional, scientific, and technical activities

We use the ILOSTAT bulk-download endpoint. If the bulk URL fails for any
reason, the cell suggests a manual fallback. Annual indicator id:
`EMP_TEMP_SEX_ECO_NB_A`.

This replaces the GGDC 10-Sector / ETD route — those products either end in
~2010 (10-Sector) or 2018 with no advanced economies (ETD), so they don't fit
a 2010–2024 global panel.


In [ ]:
import gzip

ILOSTAT_INDICATOR = "EMP_TEMP_SEX_ECO_NB_A"
ILOSTAT_URLS = [
    # Bulk download endpoints (try in order)
    f"https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/{ILOSTAT_INDICATOR}.csv.gz",
    f"https://rplumber.ilo.org/data/indicator/?id={ILOSTAT_INDICATOR}&type=both&format=.csv",
]

log.info(f"Downloading ILOSTAT employment-by-activity ({ILOSTAT_INDICATOR})...")
ilo_raw = None
for u in ILOSTAT_URLS:
    try:
        r = requests.get(u, timeout=180, headers={"User-Agent": "academic-research/1.0"})
        r.raise_for_status()
        content = r.content
        if u.endswith(".gz"):
            content = gzip.decompress(content)
        ilo_raw = pd.read_csv(BytesIO(content), low_memory=False)
        log.info(f"  got {len(ilo_raw):,} rows from {u}")
        break
    except Exception as e:
        log.warning(f"  failed {u}: {e}")

if ilo_raw is None:
    log.error("ILOSTAT download failed on all endpoints. "
              "Manual fallback: download "
              f"https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/{ILOSTAT_INDICATOR}.csv.gz "
              "and place at /content/data/raw/ilostat/EMP_TEMP_SEX_ECO_NB_A.csv.gz")
else:
    # Normalize column names
    ilo_raw.columns = [c.lower() for c in ilo_raw.columns]
    log.info(f"ILOSTAT columns: {list(ilo_raw.columns)}")

    # The bulk file has: ref_area, source, indicator, sex, classif1, time, obs_value, ...
    # classif1 looks like "ECO_ISIC4_TOTAL", "ECO_ISIC4_J", "ECO_ISIC4_K", etc.
    # We want sex=SEX_T (total), and the ISIC4 sector codes.
    ilo = ilo_raw.copy()

    # Filter to total sex
    if "sex" in ilo.columns:
        ilo = ilo[ilo["sex"].astype(str).str.upper().isin(["SEX_T", "TOTAL", "T"])]

    # Restrict to ISIC4 series only
    if "classif1" in ilo.columns:
        ilo = ilo[ilo["classif1"].astype(str).str.startswith("ECO_ISIC4_")]

    # Extract sector code (J, K, M, N, TOTAL, ...) from classif1
    ilo["sector"] = (ilo["classif1"].astype(str)
                     .str.replace("ECO_ISIC4_", "", regex=False))

    # Keep the four series we need
    KIS_SECTORS = ["J", "K", "M"]  # add "N" if you want a broader KIS definition
    keep_sectors = KIS_SECTORS + ["TOTAL"]
    ilo = ilo[ilo["sector"].isin(keep_sectors)].copy()

    # Standardize year and value
    year_col = "time" if "time" in ilo.columns else "year"
    val_col  = "obs_value" if "obs_value" in ilo.columns else "value"
    country_col = "ref_area" if "ref_area" in ilo.columns else "country"

    ilo["year"] = pd.to_numeric(ilo[year_col], errors="coerce").astype("Int64")
    ilo["emp"]  = pd.to_numeric(ilo[val_col],  errors="coerce")
    ilo = ilo.dropna(subset=["year", "emp"]).copy()
    ilo["year"] = ilo["year"].astype(int)
    ilo = ilo[(ilo["year"] >= YEAR_MIN) & (ilo["year"] <= YEAR_MAX)]

    # ref_area in ILOSTAT is already ISO3 (e.g. "USA", "GBR")
    ilo["iso3"] = ilo[country_col].astype(str).str.strip().str.upper()
    # Drop region aggregates (ILOSTAT uses things like "X01", "WORLD"); keep 3-letter codes
    ilo = ilo[ilo["iso3"].str.fullmatch(r"[A-Z]{3}")]

    # Pivot to one column per sector
    wide = (ilo.pivot_table(index=["iso3","year"], columns="sector",
                            values="emp", aggfunc="mean")
              .reset_index())
    wide.columns.name = None

    # Compute KIS share
    if "TOTAL" in wide.columns:
        kis_cols = [c for c in KIS_SECTORS if c in wide.columns]
        wide["emp_kis"] = wide[kis_cols].sum(axis=1, skipna=True)
        wide["kis_share"] = wide["emp_kis"] / wide["TOTAL"]
        # Sanity: anything > 1 or < 0 is bad
        bad = ((wide["kis_share"] < 0) | (wide["kis_share"] > 1)).sum()
        if bad:
            log.warning(f"  {bad} rows have implausible kis_share, setting to NaN")
            wide.loc[(wide["kis_share"] < 0) | (wide["kis_share"] > 1), "kis_share"] = np.nan

        # Panel only needs the share — sector employment levels stay out
        out = wide[["iso3", "year", "kis_share"]].dropna(subset=["kis_share"])
        out.to_csv(RAW/"ilostat_kis.csv", index=False)
        log.info(f"ILOSTAT KIS saved: {len(out):,} rows, "
                 f"{out['iso3'].nunique()} countries, "
                 f"mean kis_share = {out['kis_share'].mean():.3f}")
        print(out.head())
    else:
        log.error("ILOSTAT data did not include a TOTAL sector; cannot compute KIS share.")


## 8. Merge everything into a single panel

Rewritten so every source enters the panel with the **right dimensions**:

| Source | Keyed on | How |
|---|---|---|
| WDI, PWT, OpenAlex, ILOSTAT KIS, WGI, Findex | `iso3, year` | outer (panel-shaped) |
| Income group (current) | `iso3` | left (time-invariant) |
| Income group (historical) | `iso3, year` | left |
| **GARI 2024** | `iso3` | left — snapshot column `gari_total_2024` |
| **IMF AIPI** | `iso3` | left — snapshot columns `aipi_*` |
| ILO WP140 rho | `income_group` | left (group-level) |

Each file is also restricted to the listed columns, so stray helper columns
can't leak in. The result is clamped to `YEAR_MIN–YEAR_MAX`, real countries
only, one row per `(iso3, year)`.

In [ ]:
log.info("Assembling panel from available raw files...")

def load_slim(fname, usecols=None):
    fpath = RAW/fname
    if not fpath.exists():
        log.warning(f"skip missing: {fname}")
        return None
    df = pd.read_csv(fpath)
    if len(df) == 0:
        log.error(f"  {fname} exists but has 0 rows — skipping (re-run its section!)")
        return None
    if usecols:
        present = [c for c in usecols if c in df.columns]
        absent = [c for c in usecols if c not in df.columns]
        if absent:
            log.warning(f"  {fname}: expected columns missing {absent}")
        df = df[present]
    return df

# ---------- 1) country-year sources (outer merge on iso3+year) ----------
PANEL_SOURCES = [
    ("wdi_controls.csv",     None),
    ("pwt.csv",              ["iso3", "year", "human_capital_pwt"]),
    ("openalex_ai_pubs.csv", ["iso3", "year", "ai_pubs_count"]),
    ("ilostat_kis.csv",      ["iso3", "year", "kis_share"]),
    ("wgi.csv",              None),  # wgi_va..wgi_cc, already slim
    ("findex_components.csv", None),
]
panel = None
for fname, usecols in PANEL_SOURCES:
    df = load_slim(fname, usecols)
    if df is None:
        continue
    df = df.drop_duplicates(subset=["iso3", "year"])
    panel = df if panel is None else panel.merge(df, on=["iso3", "year"], how="outer")
    log.info(f"merged {fname}: panel shape = {panel.shape}")

if panel is None:
    raise RuntimeError("No country-year sources available — run Sections 3-7 first.")

# ---------- 2) time-invariant country snapshots (left merge on iso3) ----------
inc = load_slim("income_groups_current.csv", ["iso3", "income_group", "mi_status", "hi_status"])
if inc is not None:
    panel = panel.merge(inc.drop_duplicates("iso3"), on="iso3", how="left")
    log.info(f"merged income_groups_current (iso3 only): {panel.shape}")

# --- self-healing: if the AIPI file is missing, fetch it from the IMF API now ---
def _fetch_aipi_inline():
    import pycountry
    UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
          "Accept": "application/json"}
    API = "https://www.imf.org/external/datamapper/api/v1"
    valid = {c.alpha_3 for c in pycountry.countries} | {"XKX"}
    try:
        inds = requests.get(f"{API}/indicators", timeout=60, headers=UA).json().get("indicators", {})
    except Exception as e:
        log.error(f"  inline AIPI fetch failed at indicator discovery: {e}")
        return False
    codes = {k: v for k, v in inds.items()
             if str((v or {}).get("dataset", "")).upper() == "AIPI"} or \
            {"AI_PI": {"label": "AI Preparedness Index"},
             "HCLMP": {"label": "Human Capital and Labor Market Policies"},
             "IEI":   {"label": "Innovation and Economic Integration"}}
    def lab(k, l):
        ll = (l or k).lower()
        if "digital" in ll: return "digital_infra"
        if "human" in ll: return "human_capital"
        if "innovation" in ll: return "innovation"
        if "regulation" in ll or "ethic" in ll: return "regulation"
        if "preparedness" in ll or k == "AI_PI": return "total"
        return k.lower()
    frames = []
    for k, v in codes.items():
        try:
            vals = requests.get(f"{API}/{k}", timeout=60, headers=UA).json().get("values", {}).get(k, {})
        except Exception as e:
            log.warning(f"  inline AIPI fetch: {k} failed: {e}")
            continue
        rows = []
        for iso3, series in vals.items():
            iso3 = str(iso3).strip().upper()
            if iso3 not in valid or not isinstance(series, dict):
                continue
            for y in sorted((y for y in series if str(y).isdigit()), key=int, reverse=True):
                val = pd.to_numeric(pd.Series([series[y]]), errors="coerce").iloc[0]
                if pd.notna(val):
                    rows.append({"iso3": iso3, f"aipi_{lab(k, (v or {}).get('label'))}": float(val)})
                    break
        if rows:
            frames.append(pd.DataFrame(rows).drop_duplicates("iso3"))
        time.sleep(0.3)
    if not frames:
        return False
    out = frames[0]
    for f in frames[1:]:
        out = out.merge(f, on="iso3", how="outer")
    out["year"] = 2023
    out.drop_duplicates("iso3").to_csv(RAW/"imf_aipi.csv", index=False)
    log.info(f"  inline AIPI fetch OK: {len(out)} countries, "
             f"{[c for c in out.columns if c.startswith('aipi_')]}")
    return True

if not (RAW/"imf_aipi.csv").exists():
    log.warning("imf_aipi.csv missing — fetching it now from the IMF DataMapper API (Section 7e was not run)...")
    _fetch_aipi_inline()

if not (RAW/"gari_2024.csv").exists():
    log.warning("gari_2024.csv missing — RUN SECTION 7c first (it auto-extracts the Oxford "
                "GARI PDF, or reads a manual GARI_2024.xlsx). Until then the panel will "
                "have no gari_total_2024 column.")

gari = load_slim("gari_2024.csv", ["iso3", "gari_total"])
if gari is not None:
    gari = gari.rename(columns={"gari_total": "gari_total_2024"}).drop_duplicates("iso3")
    panel = panel.merge(gari, on="iso3", how="left")
    log.info(f"merged gari_2024 as snapshot (iso3 only): {panel.shape}")

aipi = load_slim("imf_aipi.csv")
if aipi is not None:
    aipi = aipi.drop(columns=[c for c in ("year",) if c in aipi.columns])
    aipi = aipi.drop_duplicates("iso3")
    panel = panel.merge(aipi, on="iso3", how="left")
    log.info(f"merged imf_aipi as snapshot (iso3 only): {panel.shape}")

# ---------- 3) country-year income history (left) ----------
hist = load_slim("income_groups_hist.csv",
                 ["iso3", "year", "income_group_hist", "mi_status_hist", "hi_status_hist"])
if hist is not None:
    panel = panel.merge(hist.drop_duplicates(["iso3", "year"]), on=["iso3", "year"], how="left")
    log.info(f"merged income_groups_hist: {panel.shape}")

# ---------- 4) income-group-level ILO rho (left, requires income_group) ----------
rho = load_slim("ilo_rho_by_income.csv")
if rho is not None and "income_group" in panel.columns:
    panel = panel.merge(rho, on="income_group", how="left")
    log.info(f"merged ilo_rho_by_income: {panel.shape}")
elif rho is not None:
    log.warning("skip ilo_rho_by_income: panel has no income_group column")

# ---------- 5) final cleanup: dimensions ----------
panel = panel.dropna(subset=["iso3"])
panel["year"] = pd.to_numeric(panel["year"], errors="coerce")
panel = panel.dropna(subset=["year"]); panel["year"] = panel["year"].astype(int)
panel = panel[(panel["year"] >= YEAR_MIN) & (panel["year"] <= YEAR_MAX)]
panel = panel[panel["iso3"].astype(str).str.fullmatch(r"[A-Z]{3}")]
before = len(panel)
panel = panel.drop_duplicates(subset=["iso3", "year"]).sort_values(["iso3", "year"]).reset_index(drop=True)
if before != len(panel):
    log.warning(f"dropped {before - len(panel)} duplicate (iso3, year) rows")

# ---------- 5b) data-quality pass: every column cut to its valid range ----------
RANGE_RULES = {
    # column-prefix or exact name : (min, max)
    "kis_share":        (0.0, 1.0),
    "gari_total_2024":  (0.0, 100.0),
    "aipi_":            (0.0, 1.0),     # AIPI and all sub-indices are 0-1 scales
    "wgi_":             (-3.5, 3.5),    # WGI estimates run roughly -2.5..+2.5
    "findex_":          (0.0, 1.0),    # Findex shares are 0-1 (or 0-100, auto-fixed below)
    "secondary_enrollment": (0.0, 200.0),  # gross enrollment can exceed 100
    "tertiary_enrollment":  (0.0, 200.0),
}
def _cols_for(rule):
    return ([rule] if rule in panel.columns
            else [c for c in panel.columns if c.startswith(rule)])

# Findex sometimes ships as 0-100 percent instead of 0-1 share — normalize first
for c in [c for c in panel.columns if c.startswith("findex_")]:
    vals = pd.to_numeric(panel[c], errors="coerce")
    if vals.max(skipna=True) is not np.nan and vals.max(skipna=True) > 1.5:
        panel[c] = vals / 100.0
        log.info(f"  {c}: detected 0-100 scale, rescaled to 0-1")

for rule, (lo, hi) in RANGE_RULES.items():
    for c in _cols_for(rule):
        vals = pd.to_numeric(panel[c], errors="coerce")
        bad = ((vals < lo) | (vals > hi)) & vals.notna()
        if bad.any():
            log.warning(f"  {c}: {int(bad.sum())} values outside [{lo}, {hi}] set to NaN")
            vals[bad] = np.nan
        panel[c] = vals

# ---------- 6) derived columns ----------
if "gdp_pc_ppp" in panel.columns:
    panel["log_gdp"] = np.log(panel["gdp_pc_ppp"])
if "ai_pubs_count" in panel.columns and panel["ai_pubs_count"].notna().any():
    panel["ai_pubs_count"] = pd.to_numeric(panel["ai_pubs_count"], errors="coerce")
    panel["ai_pubs_log"] = np.log1p(panel["ai_pubs_count"])
    mu, sd = panel["ai_pubs_log"].mean(), panel["ai_pubs_log"].std()
    panel["G_pillar_activity_starter"] = (panel["ai_pubs_log"] - mu) / sd

try:
    panel.to_parquet(PROCESSED/"panel.parquet", index=False)
    log.info(f"Panel saved as parquet: {PROCESSED/'panel.parquet'}")
except Exception:
    panel.to_csv(PROCESSED/"panel.csv", index=False)
    log.info(f"Panel saved as CSV: {PROCESSED/'panel.csv'}")

print(f"\nFinal panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"Countries: {panel['iso3'].nunique()}, years: {panel['year'].min()}-{panel['year'].max()}")
print(f"\nColumns: {list(panel.columns)}")
print("\nNon-null coverage per column:")
cov = panel.notna().mean().round(3).sort_values(ascending=False)
print(cov)
empty_cols = cov[cov == 0].index.tolist()
if empty_cols:
    print(f"\n*** WARNING: completely empty columns: {empty_cols}")
    print("    → the source section for these either failed or produced an empty file. "
          "Re-run that section, then re-run this merge.")


# ---------- 7) expected-column checklist ----------
EXPECTED = {
    "WDI (Sec 3)":            ["gdp_pc_growth", "inflation"],
    "PWT (Sec 5)":            ["human_capital_pwt"],
    "Findex (Sec 6)":         ["findex_account_t_d"],
    "OpenAlex (Sec 7)":       ["ai_pubs_count"],
    "ILO rho (Sec 7b)":       ["rho_gradient4"],
    "GARI (Sec 7c)":          ["gari_total_2024"],
    "WGI (Sec 7d)":           ["wgi_ge"],
    "IMF AIPI (Sec 7e)":      ["aipi_total"],
    "Income hist (Sec 7f)":   ["income_group_hist"],
    "ILOSTAT KIS (Sec 7g)":   ["kis_share"],
}
print("\nSource checklist:")
for src_name, cols_expected in EXPECTED.items():
    present = [c for c in cols_expected if c in panel.columns and panel[c].notna().any()]
    status = "OK " if len(present) == len(cols_expected) else "MISSING"
    print(f"  [{status}] {src_name}" + ("" if status == "OK " else
          f"  → re-run that section, then re-run this Section 8 cell"))


### Export the panel to your laptop

In [ ]:
panel.to_csv("/content/final_panel.csv", index=False)
from google.colab import files
files.download("/content/final_panel.csv")


## 9. Manual downloads — get these in your browser, then upload to Colab

After you download these files, upload them to `/content/data/raw/<subdir>/`
using the folder panel on the left (right-click → Upload). Then re-run
Section 8 to merge.

| Source | URL | Filename | Place in |
|--------|-----|----------|----------|
| **Oxford GARI 2024** | https://oxfordinsights.com/ai-readiness/ | `GARI_2024.xlsx` | `/content/data/raw/oxford_gari/` |
| **IMF AIPI** | https://www.imf.org/external/datamapper/datasets/AIPI | `AIPI_country.xlsx` | `/content/data/raw/imf_aipi/` |
| **ILO WP 140** | https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure | `ILO_WP140_exposure.xlsx` | `/content/data/raw/ilo_wp140/` |
| **GGDC 10-Sector** | https://www.rug.nl/ggdc/productivity/10-sector/ | `10SD_Database.xlsx` | `/content/data/raw/ggdc/` |
| **Findex (older waves)** | https://www.worldbank.org/en/publication/globalfindex/download-data | `GlobalFindex_<year>.xlsx` | `/content/data/raw/findex/` |
| **WGI** | https://www.worldbank.org/en/publication/worldwide-governance-indicators/interactive-data-access | `WGI_data.xlsx` | `/content/data/raw/wgi/` |
| **WB income history** | https://datahelpdesk.worldbank.org/knowledgebase/articles/906519 ("historical classification by income") | `OGHIST.xlsx` | `/content/data/raw/wb_income/` |
| **Stanford AI Index (optional)** | https://hai.stanford.edu/ai-index | `stanford_ai_index.csv` | `/content/data/raw/stanford_ai_index/` |

For each variable, see the table below for what column to extract.

In [ ]:
manual_table = pd.DataFrame([
    {"source": "Oxford GARI", "variable": "gari_government",
     "find_in_file": "Sheet 'Country Scores' column 'Government'"},
    {"source": "Oxford GARI", "variable": "gari_tech_sector",
     "find_in_file": "Sheet 'Country Scores' column 'Technology Sector'"},
    {"source": "Oxford GARI", "variable": "gari_data_infra",
     "find_in_file": "Sheet 'Country Scores' column 'Data and Infrastructure'"},
    {"source": "IMF AIPI", "variable": "aipi_digital_infra",
     "find_in_file": "Column 'Digital Infrastructure'"},
    {"source": "IMF AIPI", "variable": "aipi_human_capital",
     "find_in_file": "Column 'Human Capital and Labor Market Policies'"},
    {"source": "IMF AIPI", "variable": "aipi_innovation",
     "find_in_file": "Column 'Innovation and Economic Integration'"},
    {"source": "IMF AIPI", "variable": "aipi_regulation",
     "find_in_file": "Column 'Regulation and Ethics'"},
    {"source": "ILO WP 140", "variable": "rho_gradient4",
     "find_in_file": "Country-level table, 'Share in Gradient 4 (%)'"},
    {"source": "GGDC 10-Sector", "variable": "sigma_kis",
     "find_in_file": "Construct: sum of FIRE+CSPS employment / total non-ag employment"},
    {"source": "Findex 2011/14/17/24", "variable": "F components",
     "find_in_file": "fin45_t_d, fin11_t_d, fin22_t_d, fin10_t_d (newer waves)"},
    {"source": "WGI", "variable": "institutions",
     "find_in_file": "Government Effectiveness Estimate"},
])
manual_table

## 10. Download your data back to your laptop

After all cells finish, your panel and all raw files are in `/content/data/`.
To save them:

1. Click the folder icon (📁) on the left sidebar
2. Right-click `data` folder → Download
3. Colab will zip it and download to your computer

Or run the cell below to make a single zip:

In [ ]:
import os, shutil
src = "/content/data"
assert os.path.isdir(src) and os.listdir(src), f"{src} is missing or empty"
zip_path = shutil.make_archive("/content/data_acquisition_output", "zip", src)
print(f"Created: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
from google.colab import files
files.download("/content/data_acquisition_output.zip")

## 11. Next steps

The F-components are already in the panel as `findex_*` columns (all waves,
country totals only). The G-index inputs are `gari_total_2024`, the `aipi_*`
snapshot columns, and `G_pillar_activity_starter` from OpenAlex. The full
index-construction code lives in the companion repository under `src/construct/`:

```python
from src.construct.fintech_index import construct_fintech_index
from src.construct.ai_index import construct_ai_index

F = construct_fintech_index(panel)   # uses the findex_* columns
G = construct_ai_index(panel)        # uses gari_/aipi_/activity columns
```

The companion `joint_digital_mit_colab.ipynb` then runs the regressions on the
merged panel.